In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import numpy as np
from sklearn.impute import KNNImputer
import os 
import random
from scipy.interpolate import splrep, BSpline
import json

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.impute import KNNImputer
import matplotlib.pyplot as plt

def data_imputation_datasets(datasets_path, saving_path, imputation_column='Throughput'):
    list_imputed_dfs = []
    for path in os.listdir(datasets_path):
        path_dataset = os.path.join(datasets_path, path)
        original_df = pd.read_csv(path_dataset)
        df = original_df.copy()

        # Removing outliers for datasets that are being imputed with KNN, linear interpolation, rolling mean, and median
        df = outlier_removal(df, imputation_column)

        # Imputing with linear interpolation
        df_imputed_linear_interpolation = df.copy().infer_objects().interpolate(method='linear', limit_direction='both')
        list_imputed_dfs.append(df_imputed_linear_interpolation)
        
        # Imputing with KNN
        df_imputed_knn = impute_knn(df.copy(), imputation_column=imputation_column)
        list_imputed_dfs.append(df_imputed_knn)

        # Imputing with rolling average and median
        df_imputed_rolling_average = impute_rolling_average(df.copy(), imputation_column=imputation_column)
        df_imputed_rolling_median = impute_rolling_median(df.copy(), imputation_column=imputation_column)
        list_imputed_dfs.extend([df_imputed_rolling_average, df_imputed_rolling_median])

        # Imputing with SVD
        df_imputed_svd = impute_svd(path_dataset)
        list_imputed_dfs.append(df_imputed_svd)

        # Save files to specified directories
        save_imputed_data(saving_path, path, df_imputed_knn, df_imputed_linear_interpolation, df_imputed_rolling_average, df_imputed_rolling_median, df_imputed_svd)

    return list_imputed_dfs

def impute_knn(df, imputation_column, k=5):
    imputer = KNNImputer(n_neighbors=k)
    df[imputation_column] = imputer.fit_transform(df[[imputation_column]])
    return df

def impute_rolling_median(df, imputation_column, window_size=3):
    df[imputation_column] = df[imputation_column].fillna(df[imputation_column].rolling(window=window_size, min_periods=1).median())
    return df

def impute_rolling_average(df, imputation_column, window_size=3):
    df[imputation_column] = df[imputation_column].fillna(df[imputation_column].rolling(window=window_size, min_periods=1).mean())
    return df

def save_imputed_data(saving_path, filename, knn, linear, rolling_avg, rolling_median, svd):
    # Ensure directories exist
    for method in ["knn", "interpolacao-linear", "media-movel", "mediana-movel", "svd"]:
        os.makedirs(f'{saving_path}/{method}/', exist_ok=True)

    knn.to_csv(f'{saving_path}/knn/{filename}', index=False)
    linear.to_csv(f'{saving_path}/interpolacao-linear/{filename}', index=False)
    rolling_median.to_csv(f'{saving_path}/mediana-movel/{filename}', index=False)
    rolling_avg.to_csv(f'{saving_path}/media-movel/{filename}', index=False)
    svd.to_csv(f'{saving_path}/svd/{filename}', index=False)


def outlier_removal(df, column):
    df[column] = df[column].replace(-1, np.nan)
    values = df[column].dropna().to_numpy()

    if values.size == 0:
        print("Insufficient data for analysis.")
        return df

    values_scaled = values / np.max(values)
    thres_min, thres_max = calculate_thresholds(values_scaled)
    values_filtered = np.where((values_scaled < thres_min) | (values_scaled > thres_max), np.nan, values_scaled)
    df.loc[~df[column].isna(), column] = values_filtered * np.max(values)

    return df

def calculate_thresholds(values):
    perc_min = np.percentile(values, np.linspace(0.1, 2, 20))
    thres_min = perc_min[np.argmax(np.diff(perc_min)) + 1]

    perc_max = np.percentile(values, np.linspace(98, 100, 20))
    thres_max = perc_max[np.argmax(np.diff(perc_max)) + 1]

    return thres_min, thres_max

def impute_svd(path_dataset):
    def svd_decomposition(matrix):
        U, S, Vt = np.linalg.svd(matrix, full_matrices=True)
        return U, S, Vt

    def calculate_rmse(matrix1, matrix2):
        return np.sqrt(np.mean((matrix1 - matrix2) ** 2))

    # Load dataset and extract timestamps
    df = pd.read_csv(path_dataset)
    timestamps = df['Timestamp'] if 'Timestamp' in df.columns else None  # Extract timestamps if present
    df = outlier_removal(df, 'Throughput')
    
    # Prepare matrix for SVD
    throughput_values = df['Throughput'].interpolate(method='linear', limit_direction='both').values
    num_cols = len(throughput_values) // 28
    matrix = throughput_values[:num_cols * 28].reshape(28, num_cols)

    previous_matrix = matrix.copy()
    rmse, max_iter, iter_count = float('inf'), 300, 0

    while rmse > 1e-3 and iter_count < max_iter:
        U, S, Vt = svd_decomposition(matrix)
        variability = np.cumsum(S**2) / np.sum(S**2)

        r = np.where(variability >= 0.95)[0][0] + 1 if np.any(variability >= 0.95) else len(S)
        U_reduced, S_reduced, Vt_reduced = U[:, :r], S[:r], Vt[:r, :]

        reconstructed_matrix = (U_reduced @ np.diag(S_reduced)) @ Vt_reduced
        rmse = calculate_rmse(reconstructed_matrix, previous_matrix)
        previous_matrix = reconstructed_matrix.copy()

        iter_count += 1

    # Flatten and reformat reconstructed matrix to DataFrame with original timestamps
    throughput_imputed = reconstructed_matrix.flatten()
    final_df = pd.DataFrame({'Timestamp': timestamps[:len(throughput_imputed)], 'Throughput': throughput_imputed})
    final_df['Throughput'] = final_df['Throughput'].fillna(method='ffill').fillna(method='bfill')  # Fill any remaining NaN values

    print(f"Final RMSE: {rmse}, for file {path_dataset}")
    return final_df


In [ ]:
datasets = '../datasets/treated longest interval with failures/'

saving_imputation_datasets = '../datasets/imputed treated longest interval'

In [ ]:
data_imputation_datasets(datasets, saving_imputation_datasets)

In [ ]:
original_datasets_path = '../datasets/choosen best svd datasets'
saving_path = '../datasets/imputed choosen best svd datasets'

In [ ]:
data_imputation_datasets(original_datasets_path, saving_path)

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

# Load the JSON data from the file
with open('rmse_results.json', 'r') as f:
    json_data = json.load(f)

# Function to parse the protocol and link from the file name
def parse_file_name(file_name):
    parts = file_name.split()
    protocol = parts[1]  # Second word is the protocol
    try:
        link_index = parts.index('data') + 1
        link = parts[link_index]
    except ValueError:
        link = 'Unknown'
    return protocol, link

# Function to load and process the data
def load_data(json_data):
    data_list = []
    for file_name, techniques in json_data.items():
        protocol, link = parse_file_name(file_name)
        for imputation, metrics in techniques.items():
            throughput = metrics['Throughput']
            data_list.append({
                'File': file_name,
                'Protocol': protocol,
                'Link': link,
                'Imputation': imputation,
                'Throughput': throughput
            })
    return data_list

# Function to plot the data
def plot_data(data_list):
    df = pd.DataFrame(data_list)
    # Create a shorter label for plotting
    df['Label'] = df['Protocol'] + ' ' + df['Link']

    # Pivot the DataFrame to have 'Label' as index and 'Imputation' as columns
    pivot_df = df.pivot(index='Label', columns='Imputation', values='Throughput')

    # Plot the data
    pivot_df.plot(kind='bar', figsize=(14, 8))
    plt.xlabel('Protocol and Link')
    plt.ylabel('Throughput')
    plt.title('Throughput by Imputation Technique, Protocol, and Link')
    plt.legend(title='Imputation Technique', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

# Load and process the data
data_list = load_data(json_data)

# Plot the data
plot_data(data_list)


In [ ]:
def impute_knn(df, k=5):
    imputer = KNNImputer(n_neighbors=k)
    df['Throughput'] = imputer.fit_transform(df[['Throughput']])
    return df

def impute_rolling_median(df, window_size=3):
    while df['Throughput'].isna().any():
        rolling_median = df['Throughput'].rolling(window=window_size, min_periods=1).median()
        df['Throughput'] = df['Throughput'].fillna(rolling_median)
    return df

def impute_rolling_average(df, window_size=3):
    while df['Throughput'].isna().any():
        rolling_mean = df['Throughput'].rolling(window=window_size, min_periods=1).mean()
        df['Throughput'] = df['Throughput'].fillna(rolling_mean)
    return df

In [ ]:
caminhos_csv = []

for file in os.listdir(original_datasets_path):
    file_path = os.path.join(original_datasets_path, file)
    caminhos_csv.append(file_path)


In [ ]:
print(caminhos_csv)

In [ ]:
original_datasets_path = '../datasets/choosen best svd datasets'
saving_path = '../datasets/imputed choosen best svd datasets'

In [ ]:
resultados = {}

for caminho in caminhos_csv:
    # Carregar o arquivo
    df = pd.read_csv(caminho)

    df = outlier_removal(df, 'Throughput')
    
    # Aplicar imputação KNN
    df_knn = impute_knn(df.copy())
    
    # Aplicar mediana móvel
    df_rolling_median = impute_rolling_median(df.copy())
    
    # Aplicar média móvel
    df_rolling_average = impute_rolling_average(df.copy())

    df_interpolation = df.interpolate(method='linear', limit_direction='both')
    
    # Salvar os resultados em novos arquivos
    output_knn = caminho.replace('../datasets/choosen best svd datasets', '../datasets/imputed choosen best svd datasets/knn')
    output_median = caminho.replace('../datasets/choosen best svd datasets', '../datasets/imputed choosen best svd datasets/mediana-movel')
    output_average = caminho.replace('../datasets/choosen best svd datasets', '../datasets/imputed choosen best svd datasets/media-movel')
    output_interpolation = caminho.replace('../datasets/choosen best svd datasets', '../datasets/imputed choosen best svd datasets/interpolacao-linear')
    
    df_knn.to_csv(output_knn, index=False)
    df_rolling_median.to_csv(output_median, index=False)
    df_rolling_average.to_csv(output_average, index=False)
    df_interpolation.to_csv(output_interpolation, index=False)

    print(f"Processed file: {caminho}")